# 🥉 **Ingesta de Datos - Capa Bronze**
*Datos crudos tal como vienen de la fuente, sin transformaciones.*

In [0]:
from databricks.connect import DatabricksSession

spark = DatabricksSession.builder.getOrCreate()

# 0. Rutas de los archivos fuente
items_path   = '/Workspace/Users/raul.perez.costero@gmail.com/.bundle/Data Engineer/dev/files/databriks-repositorio/notebooks/olist_order_items_dataset.csv'
orders_path  = '/Workspace/Users/raul.perez.costero@gmail.com/.bundle/Data Engineer/dev/files/databriks-repositorio/notebooks/olist_orders_dataset.csv'
products_path = '/Workspace/Users/raul.perez.costero@gmail.com/.bundle/Data Engineer/dev/files/databriks-repositorio/notebooks/olist_products_dataset.csv'

# 1. Leer los archivos CSV
df_items = (spark
    .read
    .format('csv')
    .option('header', 'true')
    .option('inferSchema', 'true')
    .load(items_path)
    )
    
df_orders = (spark
    .read
    .format('csv')
    .option('header', 'true')
    .option('inferSchema', 'true')
    .load(orders_path)
    )

df_products = (spark
    .read
    .format('csv')
    .option('header', 'true')
    .option('inferSchema', 'true')
    .load(products_path)
    )

# 2. Crear el schema en tu catálogo 'workspace'
spark.sql("CREATE SCHEMA IF NOT EXISTS workspace.retail_db")

# 3. Guardar en formato Delta (Capa Bronce) — siempre 3 niveles: catalogo.schema.tabla
(df_orders
    .write
    .format('delta')
    .mode('overwrite')
    .saveAsTable('workspace.retail_db.orders')
    )

(df_items
    .write
    .format('delta')
    .mode('overwrite')
    .saveAsTable('workspace.retail_db.items')
)

(df_products
    .write
    .format('delta')
    .mode('overwrite')
    .saveAsTable('workspace.retail_db.products')
)

print('✅ ¡Tablas Bronce creadas correctamente en \33[35mworkspace.retail_db\33[0m! ✅')

In [0]:
# Establece el espacio de trabajo y la base de datos, para evitar tener que especificarlo en cada consulta.
# Evita: SELECT * FROM workspace.retail_db.items LIMIT 3
spark.catalog.setCurrentCatalog("workspace")
spark.catalog.setCurrentDatabase("retail_db")

In [0]:
# Consulta desde Python-SQL:
spark.sql('SELECT * FROM items LIMIT 3').show()

In [0]:
%sql
--Consulta desde SQL:
DESCRIBE items


In [0]:
%sql
SHOW TABLES


# 🥈 **Transformación de Datos - Capa Silver**
*Datos limpios, validados y estructurados listos para analizar.*

# 🥇 **Modelado de Datos - Capa Gold**
*Datos agregados y optimizados para consumo de negocio.*

# 💎 **Serving Layer - Capa Platinum** 
*Métricas y KPIs finales expuestos para dashboards y reportes.*

In [0]:
## No tiene sentido hacer esta capa en un notebook, usar Power BI + Snowflake, MLflow, Databricks Model Serving, , Tableau, Looker